# 使用LLaMA-Factory对Qwen1.5-7B模型在新闻标题生成任务上进行指令微调

## 1. 数据集
数据来源：https://github.com/percent4/UniLM_Chinese_DEMO  
需要将原始数据预处理为istruction-tuning形式，如下所示

In [ ]:
{
    "instruction": "为下面的文本生成一个合适的中文短标题。",
    "input": "中国工程院院士邬贺铨介绍，利用大数据资料可以分析人类用不同语言想表达出的意思。微软正准备开发26种语言翻译的手机，将来你打电话讲中文，海外朋友听到的就是已经翻译过的英文。大数据让未来生活充满无限可能，网友：急问，未来还有多远？未来可能不用苦苦学外语了 ",
    "output": "[话筒]未来可能不用苦苦学外语了"
}

将训练数据和测试数据加入至LLaMA-Factory下的data目录，并在data/dataset_info.json文件中加入数据集信息，如下所示

```
"weibo_sum_train": {
    "file_name": "weibo_train_data.json"
  },
  "weibo_sum_test": {
    "file_name": "weibo_test_data.json"
  }
```

## 2. 模型指令微调

```
CUDA_VISIBLE_DEVICES=0 nohup python src/train_bash.py \
    --stage sft \
    --do_train True \
    --model_name_or_path /models/Qwen1.5-7B \
    --finetuning_type lora \
    --template qwen \
    --dataset weibo_sum_train \
    --use_unsloth \
    --cutoff_len 1024 \
    --learning_rate 5e-05 \
    --num_train_epochs 1.0 \
    --max_samples 1000000 \
    --per_device_train_batch_size 16 \
    --per_device_eval_batch_size 16 \
    --gradient_accumulation_steps 4 \
    --lr_scheduler_type cosine \
    --max_grad_norm 1.0 \
    --logging_steps 1000 \
    --save_steps 1000 \
    --eval_steps 1000 \
    --evaluation_strategy steps \
    --warmup_steps 1000 \
    --output_dir saves/qwen15_7b/lora \
    --bf16 True \
    --lora_rank 8 \
    --lora_dropout 0.1 \
    --lora_target q_proj,v_proj \
    --val_size 0.2 \
    --load_best_model_at_end True \
    --plot_loss True \
    --report_to "tensorboard" &
```

## 3. 模型测试
训练后的模型文件：saves目录下  
全量测试集上进行评估命令如下
```
CUDA_VISIBLE_DEVICES=0 nohup python src/train_bash.py \
    --stage sft \
    --do_predict \
    --model_name_or_path /models/Qwen1.5-7B \
    --adapter_name_or_path saves/qwen15_7b/lora/checkpoint-6000 \
    --dataset weibo_sum_test \
    --template qwen \
    --finetuning_type lora \
    --output_dir saves/qwen15_7b/lora/output \
    --per_device_eval_batch_size 16 \
    --max_samples 1000000 \
    --predict_with_generate \
    --fp16 &
```

评估耗时约3.25小时，评估指标如下：
```
predict_bleu-4             =    26.5542
predict_rouge-1            =    39.8288
predict_rouge-2            =    23.0813
predict_rouge-l            =    37.2813
predict_runtime            = 3:16:46.63
predict_samples_per_second =     11.517
predict_steps_per_second   =       0.72
```

可以使用UniLM_Chinese_DEMO中的评估脚本来进行计算，与之前Chinese UniLM模型的评估结果进行对比